In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.6.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Lyler\Documents\biked-commons\src\biked_commons\design_evaluation\../..\biked_commons\prediction\usability_predictors.py:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data whic

In [6]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

#sample 100
data_tens = torch.tensor(data.values, dtype=torch.float32)

In [7]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(StandardEvaluations, data.columns)
isobjective = torch.tensor(requirement_types) == 1


In [8]:
num_data = data.shape[0]
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")
image_embeddings = conditioning.sample_image_embedding(num_data, split="test")
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
# condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [9]:
eval_scores = evaluator(data_tens, condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [10]:
#check gradient of eval scores wrt data_tens
data_tens.requires_grad = True
eval_scores = evaluator(data_tens, condition)
eval_scores_sum = eval_scores.sum()
eval_scores_sum.backward()
print(data_tens.grad.shape)
print(data_tens.grad[0])



c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


torch.Size([4510, 94])
tensor([-2.5020e+00,  3.1389e+00,  1.8290e+00,  8.2234e+00, -3.8681e+00,
         6.7223e-01, -6.7578e-01, -5.0839e+00, -2.5313e+00, -1.0389e+00,
         2.4403e-02, -9.4274e-02, -1.9968e-01, -1.3950e-01, -3.8901e-02,
         6.2575e-02, -6.2960e-03,  1.0023e+00,  2.9290e-03,  3.8662e+00,
        -2.4763e-02,  1.6598e-03,  8.6781e-03,  4.9274e-02, -5.6388e-01,
         2.2759e-01,  2.7016e-02, -1.6898e-01, -8.5537e-01,  3.3727e-02,
        -2.2651e+00, -5.7776e-01,  3.0813e-02, -7.3639e-02,  5.0000e-01,
         1.2343e-07,  5.0000e-01, -1.8355e-08, -3.1875e-06, -1.8470e-02,
        -3.8016e-02, -5.0477e-07,  1.5878e-02, -6.9033e-06,  1.5066e-06,
        -5.3626e-07, -3.8959e-06,  1.1844e-05,  7.8558e-07,  6.3653e-06,
         4.0495e-06,  1.0000e+00, -1.0000e+00, -1.0000e+00, -2.7596e-07,
        -4.1206e-07, -5.0346e-08,  4.2357e-09,  3.1954e-09, -5.3652e-01,
        -1.0900e-08, -1.0000e+00,  2.9670e+00, -2.7156e-05, -1.8450e-05,
        -3.0140e-05, -2.1696

In [11]:
isobjective = torch.tensor(requirement_types) == 1
objective_scores = eval_scores[:, isobjective].detach().numpy()
# constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [12]:
main_scorer = construct_scorer(MainScores, StandardEvaluations, data.columns)
detailed_scorer = construct_scorer(DetailedScores, StandardEvaluations, data.columns)

Calculating reference point for scoring functions...


c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [13]:
main_scorer(data_tens.detach(), condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Hypervolume                     0.000000e+00
Constraint Satisfaction Rate    8.585957e-01
Maximum Mean Discrepancy       -7.171762e-08
dtype: float64

In [14]:
detailed_scorer(data_tens.detach(), condition)

c:\Users\Lyler\mambaforge\envs\torch\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Min Objective Score: Usability Score - 0 to 1                                                                 0.791411
Min Objective Score: Drag Force                                                                              27.732685
Min Objective Score: Knee Angle Error                                                                       197.293720
Min Objective Score: Hip Angle Error                                                                        805.295530
Min Objective Score: Arm Angle Error                                                                        842.159670
Min Objective Score: Cosine Similarity to Embedding                                                           0.286874
Min Objective Score: Mass                                                                                    14.987815
Min Objective Score: Planar Compliance                                                                      115.509544
Min Objective Score: Transverse Compliance      